In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mssm.models import *
import pickle
import matplotlib
import copy as copy
from matplotlib import font_manager
import dotenv
import os
dotenv.load_dotenv()

# Optionally load fonts (plots want "Source Sans 3")
font_path = os.getenv("font_path")

if font_path is not None:
    font_files = font_manager.findSystemFonts(fontpaths=font_path)
    for font_file in font_files:
        font_manager.fontManager.addfont(font_file)

# Some settings for plots
cmp = matplotlib.colormaps['RdYlBu_r']
plt.rcParams["font.family"] = "Source Sans 3"
plt.rcParams["font.weight"] = "semibold"
plt.rcParams["font.size"] = 8
plt.rcParams["axes.titlesize"] = 9.5
plt.rcParams["axes.labelsize"] = 8
plt.rcParams["xtick.labelsize"] = 8
plt.rcParams["ytick.labelsize"] = 8
plt.rcParams["legend.fontsize"] = 8
plt.rcParams["figure.titlesize"] = 11
math_font_size = 8
math_font = 'cm'

seed = 42*3
np_gen = np.random.default_rng(seed)

size_conv = 2.54
single_width = 6/size_conv
double_width = 12/size_conv
full_width = 19/size_conv

## Simulations 1-3

In [ ]:
comp_cols = np.linspace(0.1,0.9,6)
n_sim = 500
sim_ids = ["sim1","sim2","sim3"]
stats = ["mse","coverage"]
cors = [True]

for sim_id in sim_ids:
    sims = [sim_id+"_exp",sim_id+"_gen"]
    sim_fam_names = [["Gaussian", "Gamma", "Binom", "Poisson"],
                     ["MGauss", "Multinomial", "ScaledT", "PropHaz"]]

    for stat in stats:

        for should_correlate in cors:
            axi = 0
            fig = plt.figure(figsize=(full_width,2*single_width),layout='constrained')
            axs = fig.subplots(2,4,gridspec_kw=dict(wspace=0.01,hspace=0.1)).flatten()
            for sim, fam_names in zip(sims,sim_fam_names):

                for fam_name in fam_names:

                    ax = axs[axi]
                    
                    try:
                        with open(f'./results/sim/{sim}/size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.pickle', 'rb') as file:
                            res = pickle.load(file)
                        res_mgcv = pd.read_csv(f'./results/sim/{sim}/mgcv_size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.csv')
                        
                        nan_idx = np.any(np.isnan(res[stat]),axis=1) | np.any(np.isnan(res_mgcv.values),axis=1)
                        print(fam_name,f"NANs (mssm, mgcv): {np.sum(np.any(np.isnan(res[stat]),axis=1) )} {np.sum(np.any(np.isnan(res_mgcv.values),axis=1))}")
                        print(f"NANs in qEFS: {np.sum(np.any(np.isnan(res[stat][:,1:]),axis=1))}")

                        res_concat = np.concat((res_mgcv[stat].values.reshape(-1,1),res[stat]),axis=1)[~nan_idx,:]
                        std_concat = (res_concat-np.median(res_concat))/np.median(res_concat)
                        ref = std_concat[:,0]

                        ax.axhline(0,linestyle='dashed',color='black',linewidth=1)
                        

                        if stat == "mse":
                            bplot = ax.boxplot(std_concat[:,1:] - ref[:,None],
                                               showfliers=True,widths=1/2,
                                               patch_artist=True,whis=1.5)
                            argmaxs = np.argmax(std_concat[:,1:] - ref[:,None],axis=0)
                            arg_max_vals = [res_concat[argmaxs[sidx],1+sidx] for sidx in range(len(argmaxs))]
                            print(fam_name,np.round(arg_max_vals,decimals=2),argmaxs)
                        elif stat == "coverage":
                            bplot = ax.boxplot(ref[:,None] - std_concat[:,1:],
                                               showfliers=True,widths=1/2,patch_artist=True,whis=1.5)

                        plt.setp(bplot['fliers'],markersize=3,alpha=0.5)

                        for pidx,(patch,line) in enumerate(zip(bplot["boxes"],bplot["medians"])):
                            patch.set_facecolor(cmp(comp_cols[pidx+1]))
                            patch.set_alpha(0.8)
                            line.set_color("black")
                    except Exception as e:
                        print(e)

                    if stat == "mse":
                        ax.set_ylabel("MSE Difference",math_fontfamily=math_font,
                                      size=math_font_size,fontweight='bold')
                    elif stat == "coverage":
                        ax.set_ylabel("Coverage Difference",math_fontfamily=math_font,
                                      size=math_font_size,fontweight='bold')

                    ylim = list(ax.get_ylim())
                    
                    if ylim[0] < -55:
                        ylim[0] = -55
                    if ylim[1] > 55:
                        ylim[1] = 55
                    ax.set_ylim(ylim)
                    

                    ax.spines['top'].set_visible(False)
                    ax.spines['right'].set_visible(False)
                    ax.spines['left'].set_visible(True)

                    ax.set_xticklabels(["EFS",
                                        "$\\mathregular{qEFS_{0}}$",
                                        "$\\mathregular{qEFS_{25}}$",
                                        "$\\mathregular{qEFS_{50}}$",
                                        "$\\mathregular{qEFS_{75}}$"])
                    
                    ax.tick_params(axis='x', labelrotation=45)
                    ax.tick_params(labelleft = True, left = True, right=False,labelright=False)
                    ax.yaxis.set_label_position("left")
                    ax.set_title(fam_name,fontweight='bold')

                    axi += 1

            fig.text(0.5,1.05,"Correlated Predictors" if should_correlate else "Uncorrelated Predictors",
                     fontweight='bold',fontsize=plt.rcParams["figure.titlesize"],ha="center", va="top",
                     transform=fig.transFigure,fontfamily="Source Sans 3")
            
            plt.savefig(f"./results/plots/{sim_id}_{stat}{"_cor" if should_correlate else ""}.pdf",
                        format="pdf", bbox_inches='tight')
            plt.show()

## Simulations 4-5

In [ ]:
sim_ids = ["sim4"]
cors = [False]
fcoef_ratios = [0.25]
comp_cols = np.linspace(0.1,0.9,2+len(fcoef_ratios))

for sim_id in sim_ids:
    sims = [sim_id+"_exp",sim_id+"_gen"]
    sim_fam_names = [["Gaussian", "Gamma", "Binom", "Poisson"],
                     ["MGauss", "Multinomial", "ScaledT", "PropHaz"]]


    for should_correlate in cors:
        axi = 0
        fig = plt.figure(figsize=(full_width,2*single_width),layout='constrained')
        axs = fig.subplots(2,4,gridspec_kw=dict(wspace=0.01,hspace=0.1)).flatten()
        for sim, fam_names in zip(sims,sim_fam_names):

            for fam_name in fam_names:

                ax = axs[axi]

                try:
                    with open(f'./results/sim/{sim}/size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.pickle', 'rb') as file:
                        res = pickle.load(file)
                    res_mgcv = pd.read_csv(f'./results/sim/{sim}/mgcv_size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.csv')

                    res_concat = np.concat((res_mgcv["aic"].values.reshape(1,-1),res["aic"]/n_sim),
                                           axis=0)
                    n_c = res_concat.shape[1]

                    ax.plot(np.linspace(0,1,n_c),res_concat[1,:],label='cAIC',
                            color="black",linestyle='solid',linewidth=1)
                    ax.plot(np.linspace(0,1,n_c),res_concat[0,:],label='WPS',
                            color=cmp(comp_cols[0]),linewidth=1)
                    ax.plot(np.linspace(0,1,n_c),res_concat[3,:],label='EFS',
                            color=cmp(comp_cols[1]),linestyle="solid",linewidth=1)
                    
                    for fci in range(len(fcoef_ratios)):
                        ax.plot(np.linspace(0,1,n_c),res_concat[4+fci,:],
                                    label="$\\mathregular{qEFS_{" + str(int(fcoef_ratios[fci]*100)) + "}}$",
                                color=cmp(comp_cols[2+fci]),linestyle="solid",linewidth=1)

                    ax.set_title(fam_name,fontweight='bold')
                    ax.spines['top'].set_visible(False)
                    ax.spines['right'].set_visible(False)
                    ax.spines['left'].set_visible(True)
                    ax.tick_params(labelleft = True, left = True, right=False,
                                   labelright=False,labelbottom=True)
                    ax.yaxis.set_label_position("left")
                    ax.set_xlabel("Effect Strength",fontweight='bold')
                    ax.set_ylabel("H0 Rejection Rate",fontweight='bold')
                    ax.set_yticks([0,0.16,0.5,1])
                    ax.set_ylim((0,1.1))
                    ax.set_xticks([0,0.5,1])
                    ax.set_xlim((0,1))

                except Exception as e:
                    print(e)
                
                axi += 1

        labels = ["cAIC", "WPS", "EFS"]
        for fci in range(len(fcoef_ratios)):
            labels.append("$\\mathregular{qEFS_{" + str(int(fcoef_ratios[fci]*100)) + "}}$")

        fig.legend(labels=labels,loc="lower center",ncol=len(labels),bbox_transform=fig.transFigure,
                    bbox_to_anchor=(0.5,-0.075),frameon=False)
        fig.text(0.5,1.05,"Smooth Term Selection" if sim_id == "sim4" else "Random Term Selection",
                    fontweight='bold',fontsize=plt.rcParams["figure.titlesize"],ha="center", va="top",
                    transform=fig.transFigure,fontfamily="Source Sans 3")

        plt.savefig(f"./results/plots/{sim_id}.pdf", format="pdf", bbox_inches='tight')
        plt.show()

        